# 15 End To End Insurance Fraud Assistant

**Project:** Insurance Fraud Detection Assistant

**Notebook:** `15-end-to-end-insurance-fraud-assistant.ipynb`

In [23]:
# ==========================================
# Notebook 15
# End To End Insurance Fraud Assistant
# ==========================================

import pandas as pd
import numpy as np

from transformers import pipeline

In [24]:
DATA_DIR = "../data"

claims_df = pd.read_csv(f"{DATA_DIR}/insurance_claims.csv")

cross_encoder_df = pd.read_csv(f"{DATA_DIR}/cross_encoder_verification.csv")

template_fraud_df = pd.read_csv(f"{DATA_DIR}/template_fraud_candidates.csv")

ring_candidates_df = pd.read_csv(f"{DATA_DIR}/semantic_ring_candidates.csv")

fraud_risk_df = pd.read_csv(f"{DATA_DIR}/fraud_risk_scores.csv")

In [25]:
print("Claims:", len(claims_df))
print("Cross Encoder:", len(cross_encoder_df))
print("Template Fraud:", len(template_fraud_df))
print("Ring Candidates:", len(ring_candidates_df))
print("Fraud Risk Scores:", len(fraud_risk_df))

Claims: 15
Cross Encoder: 16
Template Fraud: 11
Ring Candidates: 9
Fraud Risk Scores: 15


In [26]:
llm = pipeline("text2text-generation", model="google/flan-t5-base")

Device set to use cpu


In [27]:
def collect_claim_evidence(claim_id):

    claim = claims_df[claims_df["claim_id"] == claim_id]

    if len(claim) == 0:

        return None

    claim = claim.iloc[0]

    risk_record = fraud_risk_df[fraud_risk_df["claim_id"] == claim_id].iloc[0]

    cross_match = cross_encoder_df[cross_encoder_df["claim_id"] == claim_id]

    verification_score = None

    if len(cross_match) > 0:

        verification_score = float(cross_match.iloc[0]["normalized_score"])

    template_matches = len(
        template_fraud_df[
            (template_fraud_df["claim_1"] == claim_id)
            | (template_fraud_df["claim_2"] == claim_id)
        ]
    )

    ring_matches = len(
        ring_candidates_df[
            (ring_candidates_df["claim_1"] == claim_id)
            | (ring_candidates_df["claim_2"] == claim_id)
        ]
    )

    evidence = {
        "claim_id": claim_id,
        "fraud_risk_index": float(risk_record["fraud_risk_index"]),
        "risk_category": risk_record["risk_category"],
        "verification_score": verification_score,
        "template_matches": template_matches,
        "ring_matches": ring_matches,
        "claimant_statement": claim["claimant_statement"],
        "police_report": claim["police_report"],
        "adjuster_notes": claim["adjuster_notes"],
    }

    return evidence

In [28]:
collect_claim_evidence("CLM001")

{'claim_id': 'CLM001',
 'fraud_risk_index': 75.89,
 'risk_category': 'High',
 'verification_score': 0.1821472668914981,
 'template_matches': 1,
 'ring_matches': 2,
 'claimant_statement': 'A vehicle rear-ended me while I was waiting at a traffic signal.',
 'police_report': 'Witnesses confirmed another driver caused the accident.',
 'adjuster_notes': 'Section international though many movement.'}

In [29]:
def build_investigation_prompt(evidence):

    return f"""
You are an Insurance Fraud Investigator.

Analyze the following claim.

Claim ID:
{evidence['claim_id']}

Fraud Risk Index:
{evidence['fraud_risk_index']}

Risk Category:
{evidence['risk_category']}

Cross Encoder Verification:
{evidence['verification_score']}

Template Fraud Matches:
{evidence['template_matches']}

Ring Matches:
{evidence['ring_matches']}

Claimant Statement:
{evidence['claimant_statement']}

Police Report:
{evidence['police_report']}

Adjuster Notes:
{evidence['adjuster_notes']}

Provide:

1. Fraud Summary

2. Key Contradictions

3. Supporting Evidence

4. Recommended Action

"""

In [30]:
def generate_investigation_brief(claim_id):

    evidence = collect_claim_evidence(claim_id)

    prompt = build_investigation_prompt(evidence)

    response = llm(prompt, max_new_tokens=300)

    return {
        "claim_id": claim_id,
        "fraud_risk_index": evidence["fraud_risk_index"],
        "risk_category": evidence["risk_category"],
        "brief": response[0]["generated_text"],
    }

In [31]:
report = generate_investigation_brief("CLM014")

report

{'claim_id': 'CLM014',
 'fraud_risk_index': 65.0,
 'risk_category': 'High',
 'brief': 'Traffic camera footage supports claimant statement.'}

In [32]:
all_reports = []

for claim_id in claims_df["claim_id"]:

    report = generate_investigation_brief(claim_id)

    all_reports.append(report)

In [33]:
reports_df = pd.DataFrame(all_reports)

reports_df.head()

,claim_id,fraud_risk_index,risk_category,brief
0,CLM001,75.89,High,A vehicle rear-ended me while I was waiting at...
1,CLM002,57.44,Medium,The vehicle changed lanes unexpectedly and hit...
2,CLM003,64.40,High,1. Fraud Summary 2. Key Contradictions 3. Supp...
3,CLM004,57.44,Medium,The vehicle changed lanes unexpectedly and hit...
4,CLM005,25.00,Low,1. Fraud Summary 2. Key Contradictions 3. Supp...


In [34]:
investigation_queue = reports_df.sort_values(by="fraud_risk_index", ascending=False)

In [35]:
investigation_queue = reports_df.sort_values(by="fraud_risk_index", ascending=False)

In [36]:
critical_claims = investigation_queue[investigation_queue["fraud_risk_index"] >= 80]

critical_claims

,claim_id,fraud_risk_index,risk_category,brief
6,CLM007,89.76,Critical,1. Fraud Summary 2. Key Contradictions 3. Supp...
8,CLM009,84.71,Critical,The driver failed to yield and collided with m...


In [37]:
high_risk_claims = investigation_queue[investigation_queue["fraud_risk_index"] >= 60]

high_risk_claims

,claim_id,fraud_risk_index,risk_category,brief
6,CLM007,89.76,Critical,1. Fraud Summary 2. Key Contradictions 3. Supp...
8,CLM009,84.71,Critical,The driver failed to yield and collided with m...
0,CLM001,75.89,High,A vehicle rear-ended me while I was waiting at...
7,CLM008,74.33,High,The claimant was driving within the speed limi...
13,CLM014,65.00,High,Traffic camera footage supports claimant state...
2,CLM003,64.40,High,1. Fraud Summary 2. Key Contradictions 3. Supp...
12,CLM013,64.36,High,A vehicle rear-ended me while I was waiting at...


In [38]:
dashboard_df = pd.merge(claims_df, fraud_risk_df, on="claim_id")

In [39]:
dashboard_df.head()

,claim_id,claimant_name,vehicle_vin,mechanic_shop,clinic_name,lawyer,claimant_statement,police_report,adjuster_notes,medical_bill,fraud_label,verification_score,template_matches,ring_matches,fraud_risk_index,risk_category
0,CLM001,Wendy Holland,FqH15433919443,Rapid Auto Repair,Care First Clinic,Smith & Associates,A vehicle rear-ended me while I was waiting at...,Witnesses confirmed another driver caused the ...,Section international though many movement.,5072,0,0.182147,1,2,75.89,High
1,CLM002,Douglas Lara,acF49501195178,Rapid Auto Repair,Wellness Recovery Center,Anderson Legal,The vehicle changed lanes unexpectedly and hit...,Police observed damage consistent with reporte...,Budget Mrs part spend middle threat smile incr...,1541,0,0.351212,1,1,57.44,Medium
2,CLM003,Chloe Murphy,xeQ24677572737,Rapid Auto Repair,Care First Clinic,Justice Partners,I was stopped at a red light when another vehi...,Accident report indicates claimant followed tr...,Similar never box line.,20226,1,0.111990,0,2,64.40,High
3,CLM004,Jodi Reynolds MD,sPL40843321198,Rapid Auto Repair,Wellness Recovery Center,Justice Partners,The vehicle changed lanes unexpectedly and hit...,Police observed damage consistent with reporte...,Section season nor political bank.,7723,0,0.351212,1,1,57.44,Medium
4,CLM005,Elizabeth Patel,mmr35740163797,Prime Vehicle Repair,Wellness Recovery Center,Smith & Associates,I was driving through an intersection when ano...,Witnesses confirmed another driver caused the ...,Kind compare across audience society.,23376,0,1.000000,3,0,25.00,Low


In [40]:
reports_df.to_csv("../data/final_investigation_reports.csv", index=False)

In [41]:
investigation_queue.to_csv("../data/final_investigation_queue.csv", index=False)

In [42]:
dashboard_df.to_csv("../data/final_dashboard_dataset.csv", index=False)

In [43]:
print(reports_df.iloc[0]["brief"])

A vehicle rear-ended me while I was waiting at a traffic signal.
